# KDIS/HGBF Experimental Protocol — Five-Paradigm Multi-Baseline Evaluation

**Publication-grade experimental protocol** for the Knowledge-Driven Information System (KDIS) operationalized through the Hybrid Graph–Behavioral Framework (HGBF).

## Protocol Structure
| Section | Content | Dataset |
|---|---|---|
| 1 | Flow-Level Behavioral Validation (Level 1) | CIC-IDS2017 |
| 2 | Entity-Level Threshold Sensitivity Analysis | WTMC2021 |
| 3 | Five-Paradigm Multi-Baseline Comparison (RF / IF / XGB / OCSVM / KDIS) | WTMC2021 |
| 4 | UNSW-NB15 Topological Boundary Test | UNSW-NB15 |
| 5 | Export & Reporting | — |

## Key Results
- **S_out_p95 threshold**: 6,012.2 (minimum qualifying out-degree = 6,029)
- **Evaluation universe**: 19,060 source entities (8 attacker, 19,052 benign)
- **Subset relationship**: S ⊂ B ∩ RF ∩ IF ∩ XGB ∩ OCSVM (confirmed across all five paradigms)
- **Hybrid Precision**: 1.000 | **FP**: 0 | **Evidence lift**: ~2,383×
- **RCI**: 0.158 | **Jaccard lift**: ~1,162×

## Label-Withheld Protocol
Ground-truth labels are **excluded** from all detection, thresholding, and RCI computations.
Labels are applied **exclusively** for post-hoc validation.


## 0. Mount Google Drive and Import Core Libraries
Run this once at the beginning of the Colab session.


In [ ]:
from google.colab import drive
drive.mount('/content/drive')


In [ ]:
import os, glob, gc, warnings
import numpy as np
import pandas as pd
warnings.filterwarnings('ignore')


## 1. CIC-IDS2017 Flow-Level Behavioral Validation
Purpose: reproduce the Level 1 behavioral channel evaluation on the full CIC-IDS2017 consolidated Parquet dataset using the two locked temporal indicators.

Expected reported outputs:
- Precision ≈ 0.84
- Recall ≈ 0.25
- FPR ≈ 0.011


In [ ]:
import pandas as pd
import numpy as np
from sklearn.metrics import precision_score, recall_score, f1_score, confusion_matrix

df = pd.read_parquet("/content/drive/MyDrive/CIC_IDS2017/CIC_IDS2017_combined.parquet")
df.replace([np.inf, -np.inf], np.nan, inplace=True)
df.dropna(subset=["Flow_IAT_Max", "Fwd_IAT_Std", "Label"], inplace=True)

df["is_attack"] = (df["Label"] != "BENIGN").astype(int)

print(df.shape)
print(df["Label"].value_counts())


In [ ]:
# ── Section 2: Behavioral Baseline (B-only) ──

p95_iat_max = df["Flow_IAT_Max"].quantile(0.95)
p95_fwd_std = df["Fwd_IAT_Std"].quantile(0.95)

df["B"] = (
    (df["Flow_IAT_Max"] >= p95_iat_max) |
    (df["Fwd_IAT_Std"] >= p95_fwd_std)
).astype(int)

y_true = df["is_attack"]
y_pred = df["B"]

precision = precision_score(y_true, y_pred, zero_division=0)
recall = recall_score(y_true, y_pred, zero_division=0)
f1 = f1_score(y_true, y_pred, zero_division=0)

tn, fp, fn, tp = confusion_matrix(y_true, y_pred).ravel()
fpr = fp / (fp + tn) if (fp + tn) > 0 else 0
precision_manual = tp / (tp + fp) if (tp + fp) > 0 else 0

print("=== B-only Formal Evaluation ===")
print(f"Flow_IAT_Max P95: {p95_iat_max:.4f}")
print(f"Fwd_IAT_Std P95:  {p95_fwd_std:.4f}")
print(f"Flagged flows:    {df['B'].sum():,} / {len(df):,}")
print(f"TP: {tp:,}")
print(f"FP: {fp:,}")
print(f"TN: {tn:,}")
print(f"FN: {fn:,}")
print(f"Precision:       {precision_manual:.4f}")
print(f"Recall:          {recall:.4f}")
print(f"F1:              {f1:.4f}")
print(f"FPR:             {fpr:.4f}")


## 2. WTMC2021 Entity-Level Sensitivity Analysis
Purpose: reproduce the all-node entity-space validation over WTMC2021 using source and destination IPs as the full graph entity universe.

This section is the primary evidence for:
- `B_src` candidate set
- `S_out_p95` structural prominence
- `B_src ∧ S_out_p95` convergence
- threshold sensitivity across P90, P95, P97, and `B_P95 ∧ S_out_μ+2σ`


In [ ]:
# ============================================================
# KDIS / HGBF Sensitivity Analysis
# Corrected Entity Space: ALL GRAPH NODES = Src IP ∪ Dst IP
# ============================================================

import os
import glob
import numpy as np
import pandas as pd

# ============================================================
# 1) CONFIGURATION
# ============================================================

DATA_DIR = "/content/drive/MyDrive/CIC_IDS2017/dataset"
CSV_PATTERN = os.path.join(DATA_DIR, "*WorkingHours*.csv")

SRC_COL = "Src IP"
DST_COL = "Dst IP"
LABEL_COL = "Label"

BEHAVIORAL_FEATURES = [
    "Flow IAT Max",
    "Fwd IAT Std"
]

PERCENTILES = [90, 95, 97]

OUTPUT_FULL = "/content/kdis_sensitivity_all_nodes_results.csv"
OUTPUT_COMPACT = "/content/kdis_hybrid_sensitivity_all_nodes_table.csv"


# ============================================================
# 2) LOAD DATA
# ============================================================

files = sorted(glob.glob(CSV_PATTERN))

if not files:
    raise FileNotFoundError(f"No files found with pattern: {CSV_PATTERN}")

print(f"Files found: {len(files)}")
for f in files:
    print(" -", os.path.basename(f))

dfs = []
for f in files:
    part = pd.read_csv(f, low_memory=False)
    part.columns = part.columns.str.strip()
    dfs.append(part)

df = pd.concat(dfs, ignore_index=True)

print("\nTotal rows loaded:", len(df))
print("\nColumns:")
print(df.columns.tolist())


# ============================================================
# 3) VALIDATE AND CLEAN COLUMNS
# ============================================================

required_cols = [SRC_COL, DST_COL, LABEL_COL] + BEHAVIORAL_FEATURES
missing = [c for c in required_cols if c not in df.columns]

if missing:
    raise ValueError(f"Missing required columns: {missing}")

df[SRC_COL] = df[SRC_COL].astype(str).str.strip()
df[DST_COL] = df[DST_COL].astype(str).str.strip()
df[LABEL_COL] = df[LABEL_COL].astype(str).str.strip()

for col in BEHAVIORAL_FEATURES:
    df[col] = pd.to_numeric(df[col], errors="coerce")

df = df.dropna(subset=[SRC_COL, DST_COL, LABEL_COL] + BEHAVIORAL_FEATURES)

print("\nRows after cleaning:", len(df))
print("\nLabel distribution:")
print(df[LABEL_COL].value_counts())


# ============================================================
# 4) ATTACK LABEL
# ============================================================

df["is_attack"] = df[LABEL_COL].str.upper().ne("BENIGN")


# ============================================================
# 5) BUILD ALL-NODE ENTITY SPACE
# ============================================================

src_ips = pd.Series(df[SRC_COL].unique())
dst_ips = pd.Series(df[DST_COL].unique())

all_ips = pd.Index(
    pd.concat([src_ips, dst_ips], ignore_index=True).drop_duplicates(),
    name="IP"
)

entity = pd.DataFrame(index=all_ips)

# Source attacker truth
src_attacker = (
    df.groupby(SRC_COL)["is_attack"]
    .max()
    .astype(int)
    .rename("src_is_attacker")
)

# Destination victim truth
dst_victim = (
    df.groupby(DST_COL)["is_attack"]
    .max()
    .astype(int)
    .rename("dst_is_victim")
)

entity = entity.join(src_attacker, how="left")
entity = entity.join(dst_victim, how="left")

entity["src_is_attacker"] = entity["src_is_attacker"].fillna(0).astype(int)
entity["dst_is_victim"] = entity["dst_is_victim"].fillna(0).astype(int)

entity["any_attack_involved"] = (
    (entity["src_is_attacker"] == 1) |
    (entity["dst_is_victim"] == 1)
).astype(int)

# Primary truth for paper:
# source attacker only, but denominator is all graph nodes.
truth = entity["src_is_attacker"].astype(bool)

total_entities = len(entity)
total_src_attackers = int(entity["src_is_attacker"].sum())
total_dst_victims = int(entity["dst_is_victim"].sum())
total_any_involved = int(entity["any_attack_involved"].sum())
total_benign_nodes = total_entities - total_src_attackers

base_rate = total_src_attackers / total_entities if total_entities else 0

print("\n=== ALL-NODE ENTITY SPACE ===")
print(f"Total graph nodes / IP entities: {total_entities:,}")
print(f"Source attacker IPs:            {total_src_attackers:,}")
print(f"Destination victim IPs:         {total_dst_victims:,}")
print(f"Any attack-involved IPs:        {total_any_involved:,}")
print(f"Baseline src-attacker rate:     {base_rate:.8f}")


# ============================================================
# 6) BUILD UNIQUE DIRECTED EDGE GRAPH
# ============================================================

edges = (
    df[[SRC_COL, DST_COL]]
    .drop_duplicates()
    .rename(columns={SRC_COL: "src", DST_COL: "dst"})
)

print("\n=== GRAPH FROM UNIQUE DIRECTED EDGES ===")
print(f"Unique directed edges: {len(edges):,}")

out_degree = edges.groupby("src")["dst"].nunique().rename("out_degree_raw")
in_degree = edges.groupby("dst")["src"].nunique().rename("in_degree_raw")

entity = entity.join(out_degree, how="left")
entity = entity.join(in_degree, how="left")

entity["out_degree_raw"] = entity["out_degree_raw"].fillna(0).astype(int)
entity["in_degree_raw"] = entity["in_degree_raw"].fillna(0).astype(int)
entity["total_degree_raw"] = entity["out_degree_raw"] + entity["in_degree_raw"]

nonzero_out = entity.loc[entity["out_degree_raw"] > 0, "out_degree_raw"]

if nonzero_out.empty:
    raise ValueError("No nonzero out-degree values found.")


# ============================================================
# 7) METRIC FUNCTIONS
# ============================================================

def compute_metrics(pred, truth, base_rate):
    pred = pred.astype(bool)
    truth = truth.astype(bool)

    tp = int((pred & truth).sum())
    fp = int((pred & ~truth).sum())
    fn = int((~pred & truth).sum())
    tn = int((~pred & ~truth).sum())

    flagged = int(pred.sum())

    precision = tp / (tp + fp) if (tp + fp) else 0.0
    recall = tp / (tp + fn) if (tp + fn) else 0.0
    f1 = (2 * precision * recall / (precision + recall)) if (precision + recall) else 0.0
    fpr = fp / (fp + tn) if (fp + tn) else 0.0

    inside_gate_rate = tp / flagged if flagged else 0.0
    lift = inside_gate_rate / base_rate if base_rate > 0 else np.nan

    return {
        "Flagged": flagged,
        "TP": tp,
        "FP": fp,
        "FN": fn,
        "TN": tn,
        "Precision": precision,
        "Recall": recall,
        "F1": f1,
        "FPR": fpr,
        "Base_Rate": base_rate,
        "Inside_Gate_Rate": inside_gate_rate,
        "Lift": lift
    }


def jaccard_rci(S, B):
    S = S.astype(bool)
    B = B.astype(bool)

    inter = int((S & B).sum())
    union = int((S | B).sum())

    return inter / union if union else 0.0


def random_jaccard_expected(S, B):
    S = S.astype(bool)
    B = B.astype(bool)

    s_rate = S.mean()
    b_rate = B.mean()

    inter = s_rate * b_rate
    union = s_rate + b_rate - inter

    return inter / union if union > 0 else 0.0


# ============================================================
# 8) BEHAVIORAL ENTITY AGGREGATION FUNCTION
# ============================================================

def build_B_src(percentile):
    benign_df = df[df[LABEL_COL].str.upper().eq("BENIGN")]

    thresholds = {
        feat: benign_df[feat].quantile(percentile / 100.0)
        for feat in BEHAVIORAL_FEATURES
    }

    b_flow = np.zeros(len(df), dtype=bool)

    for feat in BEHAVIORAL_FEATURES:
        b_flow |= (df[feat].values >= thresholds[feat])

    temp = pd.DataFrame({
        "src": df[SRC_COL].values,
        "B_flow": b_flow.astype(int)
    })

    B_src = (
        temp.groupby("src")["B_flow"]
        .max()
        .reindex(entity.index)
        .fillna(0)
        .astype(bool)
    )

    return B_src, thresholds


# ============================================================
# 9) SENSITIVITY ANALYSIS OVER ALL GRAPH NODES
# ============================================================

results = []

for p in PERCENTILES:
    print(f"\nRunning all-node sensitivity analysis for P{p}...")

    B_src, b_thresholds = build_B_src(p)

    s_threshold = nonzero_out.quantile(p / 100.0)
    S_out = entity["out_degree_raw"] >= s_threshold

    Hybrid = B_src & S_out

    # B-only
    metrics_B = compute_metrics(B_src, truth, base_rate)
    results.append({
        "Threshold_Policy": f"P{p}",
        "Configuration": "B-only",
        "Rule": f"B_src using BENIGN P{p} temporal thresholds",
        "Behavioral_Flow_IAT_Max_Threshold": b_thresholds["Flow IAT Max"],
        "Behavioral_Fwd_IAT_Std_Threshold": b_thresholds["Fwd IAT Std"],
        "Structural_OutDegree_Threshold": np.nan,
        "RCI": np.nan,
        "RCI_Random": np.nan,
        "RCI_Lift": np.nan,
        **metrics_B
    })

    # S-only
    metrics_S = compute_metrics(S_out, truth, base_rate)
    results.append({
        "Threshold_Policy": f"P{p}",
        "Configuration": "S-only",
        "Rule": f"S_out using nonzero out-degree P{p}",
        "Behavioral_Flow_IAT_Max_Threshold": np.nan,
        "Behavioral_Fwd_IAT_Std_Threshold": np.nan,
        "Structural_OutDegree_Threshold": s_threshold,
        "RCI": np.nan,
        "RCI_Random": np.nan,
        "RCI_Lift": np.nan,
        **metrics_S
    })

    # Hybrid
    metrics_H = compute_metrics(Hybrid, truth, base_rate)

    rci = jaccard_rci(S_out, B_src)
    rci_random = random_jaccard_expected(S_out, B_src)
    rci_lift = rci / rci_random if rci_random > 0 else np.nan

    results.append({
        "Threshold_Policy": f"P{p}",
        "Configuration": "Hybrid",
        "Rule": f"B_src P{p} AND S_out P{p}",
        "Behavioral_Flow_IAT_Max_Threshold": b_thresholds["Flow IAT Max"],
        "Behavioral_Fwd_IAT_Std_Threshold": b_thresholds["Fwd IAT Std"],
        "Structural_OutDegree_Threshold": s_threshold,
        "RCI": rci,
        "RCI_Random": rci_random,
        "RCI_Lift": rci_lift,
        **metrics_H
    })


# ============================================================
# 10) μ + 2σ ROBUSTNESS CHECK
# ============================================================

print("\nRunning all-node structural μ+2σ robustness check with behavioral P95...")

B95_src, b95_thresholds = build_B_src(95)

mu = nonzero_out.mean()
sigma = nonzero_out.std()
mu2s_threshold = mu + 2 * sigma

S_mu2s = entity["out_degree_raw"] >= mu2s_threshold
Hybrid_mu2s = B95_src & S_mu2s

metrics_mu2s = compute_metrics(Hybrid_mu2s, truth, base_rate)

rci_mu2s = jaccard_rci(S_mu2s, B95_src)
rci_random_mu2s = random_jaccard_expected(S_mu2s, B95_src)
rci_lift_mu2s = rci_mu2s / rci_random_mu2s if rci_random_mu2s > 0 else np.nan

results.append({
    "Threshold_Policy": "B_P95 + S_mu+2sigma",
    "Configuration": "Hybrid",
    "Rule": "B_src P95 AND S_out μ+2σ",
    "Behavioral_Flow_IAT_Max_Threshold": b95_thresholds["Flow IAT Max"],
    "Behavioral_Fwd_IAT_Std_Threshold": b95_thresholds["Fwd IAT Std"],
    "Structural_OutDegree_Threshold": mu2s_threshold,
    "RCI": rci_mu2s,
    "RCI_Random": rci_random_mu2s,
    "RCI_Lift": rci_lift_mu2s,
    **metrics_mu2s
})


# ============================================================
# 11) RESULTS TABLES
# ============================================================

res = pd.DataFrame(results)

display_cols = [
    "Threshold_Policy",
    "Configuration",
    "Rule",
    "Flagged",
    "TP",
    "FP",
    "FN",
    "TN",
    "Precision",
    "Recall",
    "F1",
    "FPR",
    "Base_Rate",
    "Inside_Gate_Rate",
    "Lift",
    "RCI",
    "RCI_Random",
    "RCI_Lift",
    "Behavioral_Flow_IAT_Max_Threshold",
    "Behavioral_Fwd_IAT_Std_Threshold",
    "Structural_OutDegree_Threshold"
]

res = res[display_cols]

pd.set_option("display.max_columns", None)
pd.set_option("display.width", 240)

print("\n=== FULL ALL-NODE SENSITIVITY RESULTS ===")
print(res.round(6).to_string(index=False))

res.to_csv(OUTPUT_FULL, index=False)
print(f"\nSaved full results to: {OUTPUT_FULL}")


# ============================================================
# 12) PAPER-READY HYBRID TABLE
# ============================================================

compact = res[res["Configuration"] == "Hybrid"].copy()

compact = compact[
    [
        "Threshold_Policy",
        "Flagged",
        "TP",
        "FP",
        "FN",
        "Precision",
        "Recall",
        "F1",
        "FPR",
        "Base_Rate",
        "Inside_Gate_Rate",
        "Lift",
        "RCI",
        "RCI_Lift"
    ]
]

print("\n=== PAPER-READY HYBRID SENSITIVITY TABLE ===")
print(compact.round(6).to_string(index=False))

compact.to_csv(OUTPUT_COMPACT, index=False)
print(f"\nSaved compact hybrid table to: {OUTPUT_COMPACT}")


# ============================================================
# 13) HYBRID FLAGGED ENTITIES
# ============================================================

print("\n=== HYBRID FLAGGED ENTITIES BY POLICY ===")

for p in PERCENTILES:
    B_src, b_thresholds = build_B_src(p)
    s_threshold = nonzero_out.quantile(p / 100.0)
    S_out = entity["out_degree_raw"] >= s_threshold
    Hybrid = B_src & S_out

    flagged = entity.loc[
        Hybrid,
        [
            "src_is_attacker",
            "dst_is_victim",
            "any_attack_involved",
            "out_degree_raw",
            "in_degree_raw",
            "total_degree_raw"
        ]
    ].copy()

    flagged["Policy"] = f"P{p}"
    flagged = flagged.reset_index().rename(columns={"index": "IP"})

    print(f"\nPolicy P{p}:")
    if flagged.empty:
        print("No hybrid entities flagged.")
    else:
        print(flagged.to_string(index=False))


# ============================================================
# 14) SANITY CHECK AGAINST EXPECTED PAPER ENTITY SPACE
# ============================================================

print("\n=== SANITY CHECK ===")
print("If this is consistent with the paper, total graph nodes should be close to 19,062.")
print(f"Computed total graph nodes: {total_entities:,}")
print(f"Computed source attackers:  {total_src_attackers:,}")
print(f"Computed base rate:         {base_rate:.8f}")


## 3. Five-Paradigm Multi-Baseline Stability Test

**Purpose:** Evaluate whether the structural channel's precision-filtering property (S ⊂ B) holds invariantly across five heterogeneous behavioral detection paradigms:

| Paradigm | Method | Type |
|---|---|---|
| 1 | KDIS B-only (P95 temporal threshold) | Label-free threshold |
| 2 | Random Forest | Supervised ensemble |
| 3 | Isolation Forest | Unsupervised anomaly |
| 4 | XGBoost | Supervised gradient boosting |
| 5 | One-Class SVM | Unsupervised one-class |

**Protocol:** All baselines applied to WTMC2021 entity space (n = 19,060). Entity-level aggregation via max-pooling. Labels withheld during detection.

**Expected finding:** S ⊂ B ∩ RF ∩ IF ∩ XGB ∩ OCSVM across all configurations.


In [ ]:
# ================================================================
# KDIS — RF Baseline v3 (Fixed entity space + feature alignment)
# ================================================================

import os, glob, gc
import numpy as np
import pandas as pd
from sklearn.ensemble import RandomForestClassifier, IsolationForest
import warnings
warnings.filterwarnings('ignore')

CIC_PARQUET         = "/content/drive/MyDrive/CIC_IDS2017/CIC_IDS2017_combined.parquet"
WTMC_DIR            = "/content/drive/MyDrive/CIC_IDS2017/dataset/"
S_OUT_P95_THRESHOLD = 6014.6
FLOW_IAT_MAX_THRESH = 43220549.0
FWD_IAT_STD_THRESH  =  5084931.0

# ── STEP 1: Peek at WTMC columns FIRST ──────────────────────────
print("STEP 1 — Reading WTMC2021 column names...")
files = sorted(glob.glob(os.path.join(WTMC_DIR,"*WorkingHours*.csv")))
if not files:
    files = sorted(glob.glob(os.path.join(WTMC_DIR,"*.csv")))
print(f"Files: {len(files)}")

# Read only first file header to get column names
wtmc_sample = pd.read_csv(files[0], nrows=100, low_memory=False)
src_col = 'Src IP'
dst_col = 'Dst IP'
lbl_col = 'Label'

# Get numeric WTMC columns
wtmc_num_cols = [c for c in wtmc_sample.select_dtypes(include=[np.number]).columns
                 if c not in {src_col,dst_col,lbl_col}]

print(f"WTMC numeric features: {len(wtmc_num_cols)}")
print(f"Sample: {wtmc_num_cols[:5]}")

# ── STEP 2: Load CIC and find COMMON features ────────────────────
print("\nSTEP 2 — Finding common features with CIC...")
cic = pd.read_parquet(CIC_PARQUET)
cic['binary_label'] = (
    cic['Label'].str.strip().str.upper() != 'BENIGN'
).astype(int)

exclude = {'Label','binary_label','Flow ID','Src IP','Dst IP','Timestamp'}
cic_num_cols = [c for c in cic.select_dtypes(include=[np.number]).columns
                if c not in exclude]

# Normalize function
def norm(s):
    return (s.strip().lower()
             .replace(' ','').replace('_','').replace('/','')
             .replace('-','').replace('(','').replace(')',''))

cic_norm  = {norm(c): c for c in cic_num_cols}
wtmc_norm = {norm(c): c for c in wtmc_num_cols}

common_keys  = set(cic_norm.keys()) & set(wtmc_norm.keys())
cic_feat     = [cic_norm[k]  for k in sorted(common_keys)]
wtmc_feat    = [wtmc_norm[k] for k in sorted(common_keys)]

print(f"Common features: {len(common_keys)}")
if len(common_keys) < 10:
    print("WARNING: few common features — check column names")
    print(f"  CIC sample:  {list(cic_norm.keys())[:10]}")
    print(f"  WTMC sample: {list(wtmc_norm.keys())[:10]}")

# ── STEP 3: Train RF on common features ─────────────────────────
print("\nSTEP 3 — Training RF on CIC (common features only)...")
benign  = cic[cic['binary_label']==0].sample(300000, random_state=42)
attacks = cic[cic['binary_label']==1].sample(
    min(300000, int(cic['binary_label'].sum())), random_state=42)
sample  = pd.concat([benign, attacks], ignore_index=True)

X = sample[cic_feat].replace([np.inf,-np.inf], np.nan).fillna(0)
y = sample['binary_label']
print(f"Training: {len(X):,} rows × {len(cic_feat)} features")

rf = RandomForestClassifier(
    n_estimators=50, max_depth=15,
    min_samples_leaf=10, n_jobs=-1,
    random_state=42, class_weight='balanced'
)
rf.fit(X, y)
train_acc = rf.score(X, y)
print(f"Training Accuracy: {train_acc:.4f}")

del cic, sample, benign, attacks, X, y
gc.collect()
print("CIC freed ✓")

# ── STEP 4: Load full WTMC2021 ───────────────────────────────────
print("\nSTEP 4 — Loading WTMC2021...")
wtmc = pd.concat(
    [pd.read_csv(f, low_memory=False) for f in files],
    ignore_index=True
)
wtmc = wtmc.replace([np.inf,-np.inf], np.nan)
print(f"WTMC2021: {len(wtmc):,} rows")

# Attack label
wtmc['is_attack_flow'] = (
    wtmc[lbl_col].astype(str).str.strip().str.upper() != 'BENIGN'
).astype(int)

# ── STEP 5: Build entity space (ALL graph nodes: src + dst) ──────
print("\nSTEP 5 — Building entity space (all graph nodes)...")

# Get all unique IPs (source + destination = all graph nodes)
src_ips = wtmc[src_col].dropna().unique()
dst_ips = wtmc[dst_col].dropna().unique()
all_ips = pd.DataFrame({'IP': list(set(src_ips) | set(dst_ips))})

# Attacker source IPs
src_attack = (wtmc.groupby(src_col)['is_attack_flow']
                  .max().reset_index()
                  .rename(columns={src_col:'IP','is_attack_flow':'src_is_attacker'}))

entity = all_ips.merge(src_attack, on='IP', how='left').fillna(0)
entity['src_is_attacker'] = entity['src_is_attacker'].astype(int)

# Out-degree (unique directed edges per source IP)
out_deg = (wtmc.groupby([src_col,dst_col]).size()
               .reset_index().groupby(src_col).size()
               .reset_index(name='out_degree')
               .rename(columns={src_col:'IP'}))
entity = entity.merge(out_deg, on='IP', how='left').fillna(0)
entity['S_out_p95'] = (entity['out_degree'] >= S_OUT_P95_THRESHOLD).astype(int)

n_total     = len(entity)
n_attackers = int(entity['src_is_attacker'].sum())
base_rate   = n_attackers / n_total
print(f"Total graph nodes:  {n_total:,}")
print(f"Attacker source IPs:{n_attackers}")
print(f"Base rate:          {base_rate:.6f} ({base_rate*100:.4f}%)")
print(f"S_out_p95 flagged:  {entity['S_out_p95'].sum()}")

# Sanity check
assert n_total > 1000, f"Entity space too small ({n_total}) — check IP columns"

# ── STEP 6: Apply RF on WTMC flows ───────────────────────────────
print("\nSTEP 6 — Applying RF...")
X_wtmc = wtmc[wtmc_feat].copy()
X_wtmc.columns = cic_feat   # rename to match RF's training names
X_wtmc = X_wtmc.replace([np.inf,-np.inf], np.nan).fillna(0)

wtmc['RF_flag'] = rf.predict(X_wtmc)
print(f"RF flagged flows: {wtmc['RF_flag'].sum():,}")

# ── STEP 7: Apply IF ─────────────────────────────────────────────
print("Applying IF...")
iso = IsolationForest(n_estimators=100, contamination='auto',
                      random_state=42, n_jobs=-1)
iso.fit(X_wtmc)
wtmc['IF_flag'] = (iso.predict(X_wtmc) == -1).astype(int)
print(f"IF flagged flows:  {wtmc['IF_flag'].sum():,}")

# KDIS behavioral
iat_col = 'Flow IAT Max'
std_col = 'Fwd IAT Std'
wtmc['B_flag'] = (
    (pd.to_numeric(wtmc[iat_col], errors='coerce') >= FLOW_IAT_MAX_THRESH) |
    (pd.to_numeric(wtmc[std_col],  errors='coerce') >= FWD_IAT_STD_THRESH)
).astype(int)
print(f"KDIS B flagged:    {wtmc['B_flag'].sum():,}")

# ── STEP 8: Entity aggregation ───────────────────────────────────
print("\nSTEP 8 — Entity aggregation...")
for col, flag in [('RF_src','RF_flag'),('IF_src','IF_flag'),('B_src','B_flag')]:
    agg = (wtmc.groupby(src_col)[flag].max()
               .reset_index()
               .rename(columns={src_col:'IP', flag:col}))
    entity = entity.merge(agg, on='IP', how='left').fillna(0)
    entity[col] = entity[col].astype(int)

entity['KDIS_Hybrid'] = (entity['B_src']  & entity['S_out_p95']).astype(int)
entity['RF_and_S']    = (entity['RF_src'] & entity['S_out_p95']).astype(int)
entity['IF_and_S']    = (entity['IF_src'] & entity['S_out_p95']).astype(int)

for col in ['B_src','RF_src','IF_src','S_out_p95',
            'KDIS_Hybrid','RF_and_S','IF_and_S']:
    print(f"  {col}: {entity[col].sum()} flagged")

# ── STEP 9: Evaluation ───────────────────────────────────────────
print("\n" + "="*60)
print("RESULTS — Entity-Level Comparison (WTMC2021)")
print("="*60)

def evaluate(entity, col):
    y  = entity['src_is_attacker'].values
    yp = entity[col].values
    tp = int(((yp==1)&(y==1)).sum())
    fp = int(((yp==1)&(y==0)).sum())
    fn = int(((yp==0)&(y==1)).sum())
    tn = int(((yp==0)&(y==0)).sum())
    fl = int(yp.sum())
    pr = tp/fl         if fl>0        else 0
    rc = tp/n_attackers if n_attackers>0 else 0
    fpr= fp/int((1-y).sum()) if (1-y).sum()>0 else 0
    f1 = 2*pr*rc/(pr+rc) if pr+rc>0 else 0
    lift = (tp/fl)/base_rate if fl>0 else 0
    return dict(Flagged=fl, TP=tp, FP=fp, FN=fn,
                Precision=round(pr,4), Recall=round(rc,4),
                F1=round(f1,4), FPR=round(fpr,6),
                Lift=f"~{lift:,.0f}×")

configs = [
    ("KDIS B-only",    "B_src"),
    ("KDIS S-only",    "S_out_p95"),
    ("KDIS Hybrid",    "KDIS_Hybrid"),
    ("RF-only",        "RF_src"),
    ("RF ∧ S_out_p95", "RF_and_S"),
    ("IF-only",        "IF_src"),
    ("IF ∧ S_out_p95", "IF_and_S"),
]

rows = []
for name, col in configs:
    r = evaluate(entity, col); r['Configuration'] = name; rows.append(r)

df = pd.DataFrame(rows)[['Configuration','Flagged','TP','FP','FN',
                          'Precision','Recall','F1','FPR','Lift']]
print(df.to_string(index=False))

# ── STEP 10: Subset & Jaccard ────────────────────────────────────
print("\n=== SUBSET ANALYSIS ===")
B  = set(entity[entity['B_src']==1]['IP'])
S  = set(entity[entity['S_out_p95']==1]['IP'])
RF = set(entity[entity['RF_src']==1]['IP'])
IF = set(entity[entity['IF_src']==1]['IP'])

for lbl, a, b in [("S ⊂ B",S,B),("S ⊂ RF",S,RF),("S ⊂ IF",S,IF),
                   ("IF ⊂ B",IF,B),("RF ⊂ B",RF,B)]:
    print(f"  {lbl}: {'✓ TRUE' if a.issubset(b) else '✗ FALSE'}")

print("\n=== JACCARD ===")
for name, other in [("B_src",B),("RF_src",RF),("IF_src",IF)]:
    i=len(S&other); u=len(S|other)
    print(f"  Jaccard(S, {name}) = {i}/{u} = {i/u:.4f}" if u>0 else f"  {name}: empty")

print("\n=== CONVERGENT ENTITIES ===")
for name, other in [("B_src",B),("RF_src",RF),("IF_src",IF)]:
    conv=S&other
    atk=entity[entity['IP'].isin(conv)]['src_is_attacker'].sum()
    print(f"  S ∧ {name}: {len(conv)} entities | TP={atk} | FP={len(conv)-atk}")

df.to_csv('/content/rf_if_kdis_v3_results.csv', index=False)
entity.to_csv('/content/rf_if_kdis_v3_entities.csv', index=False)
print("\nSaved: rf_if_kdis_v3_results.csv")
print("Saved: rf_if_kdis_v3_entities.csv")
print("\nDone ✓")


In [ ]:
"""
KDIS — XGBoost & One-Class SVM Baselines
Adds to the entity-level comparison table from Cell 10.
Run AFTER Cell 10 (entity, wtmc, X_wtmc, cic_feat, wtmc_feat already loaded).
"""

# ================================================================
# KDIS — XGBoost & One-Class SVM Baselines
# Entity-level evaluation on WTMC2021
# Run immediately after Cell 10 (RF/IF baseline cell)
# ================================================================

import gc
import numpy as np
import pandas as pd
from sklearn.svm import OneClassSVM
from sklearn.preprocessing import StandardScaler
from xgboost import XGBClassifier
import warnings
warnings.filterwarnings('ignore')

# ── Configuration (match Cell 10) ─────────────────────────────────────────────
CIC_PARQUET         = "/content/drive/MyDrive/CIC_IDS2017/CIC_IDS2017_combined.parquet"
S_OUT_P95_THRESHOLD = 6014.6
src_col             = 'Src IP'

# ── STEP 1: Reload CIC-IDS2017 (freed in Cell 10) ────────────────────────────
print("STEP 1 — Reloading CIC-IDS2017 for XGBoost & One-Class SVM training...")
cic = pd.read_parquet(CIC_PARQUET)
cic['binary_label'] = (
    cic['Label'].str.strip().str.upper() != 'BENIGN'
).astype(int)
print(f"CIC loaded: {len(cic):,} rows")

# ── STEP 2: Training sample (same as Cell 10) ─────────────────────────────────
print("\nSTEP 2 — Preparing training data...")
benign  = cic[cic['binary_label']==0].sample(300_000, random_state=42)
attacks = cic[cic['binary_label']==1].sample(
    min(300_000, int(cic['binary_label'].sum())), random_state=42)
sample  = pd.concat([benign, attacks], ignore_index=True)

X_train = sample[cic_feat].replace([np.inf, -np.inf], np.nan).fillna(0)
y_train = sample['binary_label']
print(f"Training set: {len(X_train):,} rows × {len(cic_feat)} features")

# ── STEP 3: Train XGBoost ─────────────────────────────────────────────────────
print("\nSTEP 3 — Training XGBoost...")
xgb = XGBClassifier(
    n_estimators=100,
    max_depth=6,
    learning_rate=0.1,
    subsample=0.8,
    colsample_bytree=0.8,
    scale_pos_weight=len(y_train[y_train==0]) / len(y_train[y_train==1]),
    use_label_encoder=False,
    eval_metric='logloss',
    random_state=42,
    n_jobs=-1,
    verbosity=0
)
xgb.fit(X_train, y_train)
train_acc_xgb = xgb.score(X_train, y_train)
print(f"XGBoost Training Accuracy: {train_acc_xgb:.4f}")

# ── STEP 4: Train One-Class SVM (on benign only) ──────────────────────────────
print("\nSTEP 4 — Training One-Class SVM (benign-only)...")

# Use a subsample for OCSVM — full dataset too slow
benign_ocsvm = cic[cic['binary_label']==0].sample(
    min(50_000, int((cic['binary_label']==0).sum())), random_state=42)

X_ocsvm = benign_ocsvm[cic_feat].replace([np.inf, -np.inf], np.nan).fillna(0)

# Scale (important for OCSVM)
scaler = StandardScaler()
X_ocsvm_scaled = scaler.fit_transform(X_ocsvm)

ocsvm = OneClassSVM(
    kernel='rbf',
    nu=0.05,       # ~5% expected outlier rate
    gamma='scale'
)
ocsvm.fit(X_ocsvm_scaled)
print("One-Class SVM trained ✓")

# Free CIC
del cic, sample, benign, attacks, X_train, benign_ocsvm, X_ocsvm, X_ocsvm_scaled
gc.collect()
print("CIC freed ✓")

# ── STEP 5: Apply to WTMC2021 flows ──────────────────────────────────────────
print("\nSTEP 5 — Applying XGBoost and One-Class SVM to WTMC2021...")

# Reuse X_wtmc from Cell 10 (already computed, columns renamed to cic_feat)
# XGBoost
wtmc['XGB_flag'] = xgb.predict(X_wtmc)
print(f"XGBoost flagged flows: {wtmc['XGB_flag'].sum():,}")

# One-Class SVM
X_wtmc_scaled = scaler.transform(
    X_wtmc.replace([np.inf, -np.inf], np.nan).fillna(0)
)
wtmc['OCSVM_flag'] = (ocsvm.predict(X_wtmc_scaled) == -1).astype(int)
print(f"One-Class SVM flagged flows: {wtmc['OCSVM_flag'].sum():,}")

del X_wtmc_scaled
gc.collect()

# ── STEP 6: Entity-level aggregation ─────────────────────────────────────────
print("\nSTEP 6 — Entity-level aggregation...")

for col, flag in [('XGB_src', 'XGB_flag'), ('OCSVM_src', 'OCSVM_flag')]:
    agg = (wtmc.groupby(src_col)[flag].max()
               .reset_index()
               .rename(columns={src_col: 'IP', flag: col}))
    entity = entity.merge(agg, on='IP', how='left').fillna(0)
    entity[col] = entity[col].astype(int)

entity['XGB_and_S']   = (entity['XGB_src']   & entity['S_out_p95']).astype(int)
entity['OCSVM_and_S'] = (entity['OCSVM_src'] & entity['S_out_p95']).astype(int)

print(f"  XGB_src flagged:      {entity['XGB_src'].sum()}")
print(f"  OCSVM_src flagged:    {entity['OCSVM_src'].sum()}")
print(f"  XGB ∧ S_out_p95:      {entity['XGB_and_S'].sum()}")
print(f"  OCSVM ∧ S_out_p95:    {entity['OCSVM_and_S'].sum()}")

# ── STEP 7: Evaluation ───────────────────────────────────────────────────────
print("\n" + "="*65)
print("RESULTS — Extended Entity-Level Comparison (WTMC2021)")
print("="*65)

new_configs = [
    ("XGB-only",          "XGB_src"),
    ("XGB ∧ S_out_p95",   "XGB_and_S"),
    ("OCSVM-only",        "OCSVM_src"),
    ("OCSVM ∧ S_out_p95", "OCSVM_and_S"),
]

rows = []
for name, col in new_configs:
    r = evaluate(entity, col)
    r['Configuration'] = name
    rows.append(r)

df_new = pd.DataFrame(rows)[['Configuration','Flagged','TP','FP','FN',
                               'Precision','Recall','F1','FPR','Lift']]
print(df_new.to_string(index=False))

# ── STEP 8: Full comparison table (all baselines) ────────────────────────────
print("\n" + "="*65)
print("FULL COMPARISON TABLE — All Configurations")
print("="*65)

all_configs = [
    ("KDIS B-only",       "B_src"),
    ("KDIS S-only",       "S_out_p95"),
    ("KDIS Hybrid",       "KDIS_Hybrid"),
    ("RF-only",           "RF_src"),
    ("RF ∧ S_out_p95",    "RF_and_S"),
    ("IF-only",           "IF_src"),
    ("IF ∧ S_out_p95",    "IF_and_S"),
    ("XGB-only",          "XGB_src"),
    ("XGB ∧ S_out_p95",   "XGB_and_S"),
    ("OCSVM-only",        "OCSVM_src"),
    ("OCSVM ∧ S_out_p95", "OCSVM_and_S"),
]

all_rows = []
for name, col in all_configs:
    r = evaluate(entity, col)
    r['Configuration'] = name
    all_rows.append(r)

df_full = pd.DataFrame(all_rows)[['Configuration','Flagged','TP','FP','FN',
                                    'Precision','Recall','F1','FPR','Lift']]
print(df_full.to_string(index=False))

# ── STEP 9: Subset verification ──────────────────────────────────────────────
print("\n=== SUBSET ANALYSIS (S ⊂ X) ===")
S   = set(entity[entity['S_out_p95']==1]['IP'])
XGB = set(entity[entity['XGB_src']==1]['IP'])
OCS = set(entity[entity['OCSVM_src']==1]['IP'])

for lbl, other in [("S ⊂ XGB", XGB), ("S ⊂ OCSVM", OCS)]:
    print(f"  {lbl}: {'✓ TRUE' if S.issubset(other) else '✗ FALSE'}")

print("\n=== JACCARD (S, new baselines) ===")
for name, other in [("XGB_src", XGB), ("OCSVM_src", OCS)]:
    i = len(S & other); u = len(S | other)
    rci = i/u if u > 0 else 0
    rci_random = (len(S)/len(entity)) * (len(other)/len(entity))
    lift = rci/rci_random if rci_random > 0 else 0
    print(f"  Jaccard(S, {name}) = {i}/{u} = {rci:.4f} | RCI Lift ≈ {lift:,.0f}×")

# Save
df_full.to_csv('/content/kdis_full_comparison_v4.csv', index=False)
entity.to_csv('/content/kdis_entities_v4.csv',         index=False)
print("\nSaved: kdis_full_comparison_v4.csv")
print("Saved: kdis_entities_v4.csv")
print("\nDone ✓")


## 4. UNSW-NB15 Mini External Boundary Validation
Purpose: run a limited portability / boundary-condition check on raw zipped UNSW-NB15 files.

This section is **not** a full external benchmark. It is used to characterize graph-topological boundary conditions, especially the effect of low entity diversity and near-uniform source out-degree distributions.


In [ ]:
# ============================================================
# UNSW-NB15 Mini External Validation for KDIS / HGBF
# Reads zipped raw UNSW files directly from MyDrive
# ============================================================

import os
import glob
import numpy as np
import pandas as pd

# ============================================================
# 1) CONFIGURATION
# ============================================================

ZIP_DIR = "/content/drive/MyDrive/UNSW_NB15"

MAX_ROWS = 200_000
RANDOM_STATE = 42
PERCENTILES = [90, 95, 97]

OUTPUT_FULL = "/content/unsw_kdis_mini_validation_full.csv"
OUTPUT_HYBRID = "/content/unsw_kdis_mini_validation_hybrid.csv"

UNSW_COLS = [
    "srcip", "sport", "dstip", "dsport", "proto", "state", "dur",
    "sbytes", "dbytes", "sttl", "dttl", "sloss", "dloss", "service",
    "Sload", "Dload", "Spkts", "Dpkts", "swin", "dwin", "stcpb", "dtcpb",
    "smeansz", "dmeansz", "trans_depth", "res_bdy_len", "Sjit", "Djit",
    "Stime", "Ltime", "Sintpkt", "Dintpkt", "tcprtt", "synack", "ackdat",
    "is_sm_ips_ports", "ct_state_ttl", "ct_flw_http_mthd", "is_ftp_login",
    "ct_ftp_cmd", "ct_srv_src", "ct_srv_dst", "ct_dst_ltm", "ct_src_ltm",
    "ct_src_dport_ltm", "ct_dst_sport_ltm", "ct_dst_src_ltm",
    "attack_cat", "Label"
]

SRC_COL = "srcip"
DST_COL = "dstip"
LABEL_COL = "Label"

features = [
    "dur",
    "Sintpkt",
    "Dintpkt",
    "Sjit",
    "Djit",
    "Sload",
    "Dload"
]

# ============================================================
# 2) LOAD ZIPPED UNSW RAW CSV FILES
# ============================================================

zip_files = sorted(glob.glob(os.path.join(ZIP_DIR, "*.zip")))

if not zip_files:
    raise FileNotFoundError(f"No ZIP files found in: {ZIP_DIR}")

print(f"ZIP files found: {len(zip_files)}")
for f in zip_files:
    print(" -", os.path.basename(f))

dfs = []

for f in zip_files:
    print(f"\nLoading: {os.path.basename(f)}")

    part = pd.read_csv(
        f,
        compression="zip",
        header=None,
        names=UNSW_COLS,
        low_memory=False,
        on_bad_lines="skip"
    )

    print("Rows:", len(part))
    dfs.append(part)

df = pd.concat(dfs, ignore_index=True)

print("\nFinal rows loaded:", len(df))
print("Columns:")
print(df.columns.tolist())

print("\nLabel distribution:")
print(df[LABEL_COL].value_counts(dropna=False))

print("\nAttack category distribution:")
print(df["attack_cat"].value_counts(dropna=False))

# ============================================================
# 3) SAMPLE FOR MINI VALIDATION
# ============================================================

if len(df) > MAX_ROWS:
    df = df.sample(n=MAX_ROWS, random_state=RANDOM_STATE).reset_index(drop=True)

print("\nRows used for mini-validation:", len(df))

# ============================================================
# 4) CLEAN
# ============================================================

df[SRC_COL] = df[SRC_COL].astype(str).str.strip()
df[DST_COL] = df[DST_COL].astype(str).str.strip()

df[LABEL_COL] = pd.to_numeric(df[LABEL_COL], errors="coerce").fillna(0).astype(int)
df["is_attack"] = df[LABEL_COL].eq(1)

for col in features:
    df[col] = pd.to_numeric(df[col], errors="coerce")

df = df.dropna(subset=[SRC_COL, DST_COL] + features)

print("\nRows after cleaning:", len(df))
print("\nAttack distribution after sampling/cleaning:")
print(df["is_attack"].value_counts())

# ============================================================
# 5) ENTITY SPACE
# ============================================================

all_ips = pd.Index(
    pd.concat(
        [
            pd.Series(df[SRC_COL].unique()),
            pd.Series(df[DST_COL].unique())
        ],
        ignore_index=True
    ).drop_duplicates(),
    name="IP"
)

entity = pd.DataFrame(index=all_ips)

src_attacker = (
    df.groupby(SRC_COL)["is_attack"]
    .max()
    .astype(int)
    .rename("src_is_attacker")
)

dst_victim = (
    df.groupby(DST_COL)["is_attack"]
    .max()
    .astype(int)
    .rename("dst_is_victim")
)

entity = entity.join(src_attacker, how="left")
entity = entity.join(dst_victim, how="left")

entity["src_is_attacker"] = entity["src_is_attacker"].fillna(0).astype(int)
entity["dst_is_victim"] = entity["dst_is_victim"].fillna(0).astype(int)

entity["any_attack_involved"] = (
    (entity["src_is_attacker"] == 1) |
    (entity["dst_is_victim"] == 1)
).astype(int)

truth = entity["src_is_attacker"].astype(bool)

total_entities = len(entity)
total_attackers = int(entity["src_is_attacker"].sum())
base_rate = total_attackers / total_entities if total_entities else 0

print("\n=== UNSW Entity Space ===")
print(f"Total IP entities:      {total_entities:,}")
print(f"Source attacker IPs:    {total_attackers:,}")
print(f"Base attacker rate:     {base_rate:.8f}")

# ============================================================
# 6) GRAPH CONSTRUCTION
# ============================================================

edges = (
    df[[SRC_COL, DST_COL]]
    .drop_duplicates()
    .rename(columns={SRC_COL: "src", DST_COL: "dst"})
)

out_degree = edges.groupby("src")["dst"].nunique().rename("out_degree_raw")
in_degree = edges.groupby("dst")["src"].nunique().rename("in_degree_raw")

entity = entity.join(out_degree, how="left")
entity = entity.join(in_degree, how="left")

entity["out_degree_raw"] = entity["out_degree_raw"].fillna(0).astype(int)
entity["in_degree_raw"] = entity["in_degree_raw"].fillna(0).astype(int)
entity["total_degree_raw"] = entity["out_degree_raw"] + entity["in_degree_raw"]

nonzero_out = entity.loc[entity["out_degree_raw"] > 0, "out_degree_raw"]

if nonzero_out.empty:
    raise ValueError("No nonzero out-degree values found.")

print("\n=== Graph Summary ===")
print(f"Unique directed edges: {len(edges):,}")
print(f"Nonzero source nodes:  {len(nonzero_out):,}")

# ============================================================
# 7) METRIC FUNCTIONS
# ============================================================

def compute_metrics(pred, truth, base_rate):
    pred = pred.astype(bool)
    truth = truth.astype(bool)

    tp = int((pred & truth).sum())
    fp = int((pred & ~truth).sum())
    fn = int((~pred & truth).sum())
    tn = int((~pred & ~truth).sum())

    flagged = int(pred.sum())

    precision = tp / (tp + fp) if (tp + fp) else 0
    recall = tp / (tp + fn) if (tp + fn) else 0
    f1 = 2 * precision * recall / (precision + recall) if (precision + recall) else 0
    fpr = fp / (fp + tn) if (fp + tn) else 0

    inside_gate_rate = tp / flagged if flagged else 0
    lift = inside_gate_rate / base_rate if base_rate > 0 else np.nan

    return {
        "Flagged": flagged,
        "TP": tp,
        "FP": fp,
        "FN": fn,
        "TN": tn,
        "Precision": precision,
        "Recall": recall,
        "F1": f1,
        "FPR": fpr,
        "Base_Rate": base_rate,
        "Inside_Gate_Rate": inside_gate_rate,
        "Lift": lift
    }


def jaccard_rci(S, B):
    S = S.astype(bool)
    B = B.astype(bool)

    inter = int((S & B).sum())
    union = int((S | B).sum())

    return inter / union if union else 0


def random_jaccard_expected(S, B):
    S = S.astype(bool)
    B = B.astype(bool)

    s_rate = S.mean()
    b_rate = B.mean()

    inter = s_rate * b_rate
    union = s_rate + b_rate - inter

    return inter / union if union > 0 else 0

# ============================================================
# 8) BEHAVIORAL ENTITY FUNCTION
# ============================================================

def build_B_src(percentile):
    benign_df = df[~df["is_attack"]]

    thresholds = {
        feat: benign_df[feat].quantile(percentile / 100.0)
        for feat in features
    }

    b_flow = np.zeros(len(df), dtype=bool)

    for feat in features:
        b_flow |= df[feat].values >= thresholds[feat]

    temp = pd.DataFrame({
        "src": df[SRC_COL].values,
        "B_flow": b_flow.astype(int)
    })

    B_src = (
        temp.groupby("src")["B_flow"]
        .max()
        .reindex(entity.index)
        .fillna(0)
        .astype(bool)
    )

    return B_src, thresholds

# ============================================================
# 9) MINI VALIDATION
# ============================================================

results = []

for p in PERCENTILES:
    print(f"\nRunning UNSW mini-validation P{p}...")

    B_src, thresholds = build_B_src(p)

    s_threshold = nonzero_out.quantile(p / 100.0)
    S_out = entity["out_degree_raw"] >= s_threshold

    Hybrid = B_src & S_out

    rci = jaccard_rci(S_out, B_src)
    rci_random = random_jaccard_expected(S_out, B_src)
    rci_lift = rci / rci_random if rci_random > 0 else np.nan

    for name, pred in {
        "UNSW B-only": B_src,
        "UNSW S-only": S_out,
        "UNSW Hybrid": Hybrid
    }.items():

        metrics = compute_metrics(pred, truth, base_rate)

        results.append({
            "Dataset": "UNSW-NB15 mini",
            "Threshold": f"P{p}",
            "Configuration": name,
            "RCI": rci if name == "UNSW Hybrid" else np.nan,
            "RCI_Random": rci_random if name == "UNSW Hybrid" else np.nan,
            "RCI_Lift": rci_lift if name == "UNSW Hybrid" else np.nan,
            "Behavioral_Features": ", ".join(features),
            "Structural_Threshold": s_threshold,
            **metrics
        })

res = pd.DataFrame(results)

pd.set_option("display.max_columns", None)
pd.set_option("display.width", 240)

print("\n=== UNSW MINI-VALIDATION RESULTS ===")
print(res.round(6).to_string(index=False))

res.to_csv(OUTPUT_FULL, index=False)
print(f"\nSaved full results to: {OUTPUT_FULL}")

# ============================================================
# 10) PAPER-READY HYBRID TABLE
# ============================================================

hybrid = res[res["Configuration"] == "UNSW Hybrid"].copy()

hybrid_table = hybrid[
    [
        "Threshold",
        "Flagged",
        "TP",
        "FP",
        "Precision",
        "Recall",
        "F1",
        "FPR",
        "Lift",
        "RCI",
        "RCI_Lift"
    ]
]

print("\n=== PAPER-READY HYBRID TABLE ===")
print(hybrid_table.round(6).to_string(index=False))

hybrid_table.to_csv(OUTPUT_HYBRID, index=False)
print(f"\nSaved hybrid table to: {OUTPUT_HYBRID}")

# ============================================================
# 11) FLAGGED HYBRID ENTITIES
# ============================================================

print("\n=== UNSW HYBRID FLAGGED ENTITIES BY THRESHOLD ===")

for p in PERCENTILES:
    B_src, _ = build_B_src(p)
    s_threshold = nonzero_out.quantile(p / 100.0)
    S_out = entity["out_degree_raw"] >= s_threshold
    Hybrid = B_src & S_out

    flagged = entity.loc[
        Hybrid,
        [
            "src_is_attacker",
            "dst_is_victim",
            "any_attack_involved",
            "out_degree_raw",
            "in_degree_raw",
            "total_degree_raw"
        ]
    ].copy()

    flagged["Threshold"] = f"P{p}"
    flagged = flagged.reset_index().rename(columns={"index": "IP"})

    print(f"\nThreshold P{p}:")
    if flagged.empty:
        print("No hybrid entities flagged.")
    else:
        print(flagged.to_string(index=False))

# ============================================================
# 12) INTERPRETATION HINT
# ============================================================

print("\n=== Interpretation Hint ===")
print("""
Use this only as:
- mini external validation
- portability sanity check

NOT as:
- full benchmark replication

Paper-safe wording:

A limited external validation experiment on UNSW-NB15 was conducted to evaluate whether sparse structural–behavioral convergence remains observable outside the CIC-derived traffic topology. The objective was not to reproduce identical RCI or recall values, but to examine whether the corroborative convergence behavior remains detectable under a different traffic environment.
""")


## 5. Final Reporting Notes
Use the exported CSV files and printed tables as the canonical experimental record. Do not cite exploratory cells from the original notebook. For thesis/article reporting, use only:

- Flow-level behavioral validation from Section 1.
- WTMC all-node sensitivity results from Section 2.
- RF/IF/KDIS cross-method comparison from Section 3.
- UNSW mini-validation as a boundary-condition result from Section 4.

Recommended artifact outputs:
- `/content/kdis_sensitivity_all_nodes_results.csv`
- `/content/kdis_hybrid_sensitivity_all_nodes_table.csv`
- `/content/rf_if_kdis_v3_results.csv`
- `/content/rf_if_kdis_v3_entities.csv`
- `/content/unsw_kdis_mini_validation_full.csv`
- `/content/unsw_kdis_mini_validation_hybrid.csv`


In [ ]:
from google.colab import files
files.download('/content/kdis_full_comparison_v4.csv')
files.download('/content/kdis_entities_v4.csv')